## 0. Path bootstrap

In [1]:
import sys
from pathlib import Path

# Ensure THIS directory is on the path so local modules are importable.
NOTEBOOK_DIR = Path().resolve()  # current working directory when running the notebook
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"Working dir : {NOTEBOOK_DIR}")

Working dir : C:\Users\vimal\OneDrive\Documents\Uni\BTP\User-Adaptive-XAI\MCC_CD_Final


In [2]:
# import torch

# torch.cuda.empty_cache()

## 1. Imports

In [3]:
import warnings

from IPython.display import display
import pandas as pd

from config import (
    # Experiment knobs
    ABLATION_MODE,
    EXPERIMENT_RESULTS_PATH,
    EXPERIMENT_TAG,
    INPUT_TEXTS,  # ← now a list
    USER_CATEGORY,
    # XAI method  ← change XAI_METHOD in config.py to swap
    XAI_METHOD,
    XAI_NUM_FEATURES,
    XAI_NUM_SAMPLES,
    CLASS_NAMES,
    # LLM / decoding
    LAMBDA_MAP,
    NUM_BEAMS,
    USE_CONSTRAINED_DECODING,
    # Ontology
    TOP_LIME_FEATURES,
)
from constrained_decoding import ReadabilityBeamGenerator
from model_loaders import load_classifier, load_llm, load_ontology_model
from pipeline_helpers import (
    enrich_with_ontology,
    generate_explanation,
    lime_coverage,
    ontology_hit_rate,
    predict_class,
    readability_metrics,
    run_xai,
)

warnings.filterwarnings("ignore")
print("✅ Imports complete.")

c:\Users\vimal\gpu_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports complete.


## 2. Experiment configuration (read-only — edit config.py)

In [4]:
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  EXPERIMENT CONFIG")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Tag                  : {EXPERIMENT_TAG}")
print(f"  XAI method           : {XAI_METHOD}")
print(f"  User category        : {USER_CATEGORY}")
print(f"  Ablation mode        : {ABLATION_MODE}")
print(f"  Constrained decoding : {USE_CONSTRAINED_DECODING}")
print(f"  Lambda value         : {LAMBDA_MAP.get(USER_CATEGORY, 'N/A')}")
print(f"  Num beams            : {NUM_BEAMS}")
print(f"  Output path          : {EXPERIMENT_RESULTS_PATH}")
n_inputs = len(INPUT_TEXTS) if INPUT_TEXTS else '(all lines in test_data.txt)'
print(f"  # Inputs             : {n_inputs}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EXPERIMENT CONFIG
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Tag                  : expert_normal_ig
  XAI method           : IG
  User category        : EXPERT
  Ablation mode        : normal
  Constrained decoding : True
  Lambda value         : 0.0
  Num beams            : 4
  Output path          : Results_w_CD\expert_normal_ig.csv
  # Inputs             : 6
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 3. Input texts

In [5]:
# ── Resolve input list ────────────────────────────────────────────────────────
if INPUT_TEXTS is not None:
    input_texts = [t.strip() for t in INPUT_TEXTS if t.strip()]
    print(f"[Input] Using {len(input_texts)} text(s) from config.py INPUT_TEXTS.")
else:
    data_path = NOTEBOOK_DIR / "test_data.txt"
    with open(data_path, encoding="utf-8") as f:
        input_texts = [ln.strip() for ln in f if ln.strip()]
    print(f"[Input] INPUT_TEXTS is None — loaded {len(input_texts)} lines from test_data.txt.")

for i, t in enumerate(input_texts, 1):
    preview = t[:120] + ("…" if len(t) > 120 else "")
    print(f"  [{i}] ({len(t)} chars) {preview}")

[Input] Using 6 text(s) from config.py INPUT_TEXTS.
  [1] (462 chars) Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with…
  [2] (670 chars) Ultrasound-Doppler diagnosis of Budd-Chiari syndrome. We report a case of apparently idiopathic Budd-Chiari syndrome, di…
  [3] (1795 chars) Neurogenic inflammation of the rat trachea: fate of neutrophils that adhere to venules. The goal of this study was to de…
  [4] (590 chars) Aberrant regeneration in a case of syringobulbia: selective co-activation of abducens and facial nerves during saccades.…
  [5] (434 chars) Germ cell tumor of testis in a patient with von Hippel-Lindau disease. Germ cell testicular tumor is a previously undesc…
  [6] (1095 chars) Failure of hepatitis B immunization in liver transplant recipients: results of a prospective trial. Twenty patients with…


## 4. Load models (classifier, ontology, LLM)

This is the slow step. All models are loaded once here and reused across all inputs.
NER is not loaded — LIME/IG runs directly on raw text.

In [6]:
# ── Classifier ────────────────────────────────────────────────────────────────
clf_model, clf_pipeline = load_classifier()

# ── Ontology ──────────────────────────────────────────────────────────────────
ontology = load_ontology_model()

# ── LLM ───────────────────────────────────────────────────────────────────────
llm_tokenizer, llm_model = load_llm()

# ── Constrained-decoding generator ────────────────────────────────────────────
generator = None
if USE_CONSTRAINED_DECODING:
    generator = ReadabilityBeamGenerator(
        model=llm_model,
        tokenizer=llm_tokenizer,
        num_beams=NUM_BEAMS,
    )
    print(f"[CD] Constrained decoding enabled — λ={LAMBDA_MAP.get(USER_CATEGORY, '?')}, beams={NUM_BEAMS}")
else:
    print("[CD] Constrained decoding disabled (greedy/sample mode).")

print("\n✅ All models loaded.")

[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4717.55it/s]


[Loader] Classifier ready.

[Ontology] Loading from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Ontology/doid.owl' …
[Ontology] Loaded successfully.
[Ontology] Stats → classes: 14493, est. max depth: 15
[Loader] Loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' …


Loading weights: 100%|██████████| 338/338 [00:08<00:00, 41.72it/s]


[Loader] LLM loaded in 4-bit quantized mode (bitsandbytes).
[Loader] LLM ready.

[CD] Constrained decoding enabled — λ=0.0, beams=4

✅ All models loaded.


## Stages 1-4 — Batch pipeline

Each input in `INPUT_TEXTS` is processed through:
- **Stage 1**: XAI feature attribution (LIME or IG)
- **Stage 2**: Ontology enrichment
- **Stage 3**: LLM explanation generation
- **Stage 4**: Readability & faithfulness metrics

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Batch pipeline  — Stages 1-4 repeated for every input text
# ══════════════════════════════════════════════════════════════════════════════

all_results = []  # collects one dict per input

for idx, input_text in enumerate(input_texts, start=1):
    print(f"\n{'='*64}")
    print(f"  Processing input {idx}/{len(input_texts)}")
    print(f"  Preview: {input_text[:80]}…")
    print(f"{'='*64}")

    # ── Stage 1 : XAI feature attribution ──────────────────────────────────────
    print(f"\n[Stage 1] Running {XAI_METHOD} feature attribution …")
    xai_features = run_xai(
        method=XAI_METHOD,
        text=input_text,
        clf_model=clf_model,
        clf_pipeline=clf_pipeline,
        class_names=CLASS_NAMES,
        num_features=XAI_NUM_FEATURES,
        num_samples=XAI_NUM_SAMPLES,
    )
    lime_features = xai_features
    print(f"[Stage 1] Top {len(lime_features)} {XAI_METHOD} features:")
    for word, score in lime_features:
        print(f"  {word:25s}  score={score:+.4f}")

    # ── Stage 2 : Ontology enrichment ──────────────────────────────────────────
    print("\n[Stage 2] Classifying text …")
    predicted_class, confidence = predict_class(input_text, clf_pipeline)
    print(f"  Predicted class : {predicted_class}")
    print(f"  Confidence      : {confidence:.4f}")

    print("\n[Stage 2] Mapping features to ontology ancestors …")
    feature_data = enrich_with_ontology(
        lime_features=lime_features,
        ontology=ontology,
        user_category=USER_CATEGORY,
        ablation_mode=ABLATION_MODE,
    )
    hit_words = [f["feature_word"] for f in feature_data]
    print(f"  Ontology hits : {hit_words} ({len(hit_words)}/{len(lime_features[:TOP_LIME_FEATURES])} features)")
    for f in feature_data:
        print(f"    {f['feature_word']:20s} → {f['ancestors']}")

    # ── Stage 3 : LLM explanation ───────────────────────────────────────────────
    print("\n[Stage 3] Building prompt …")
    if USE_CONSTRAINED_DECODING:
        lam = LAMBDA_MAP.get(USER_CATEGORY, LAMBDA_MAP["EXPERT"])
        print(f"[Stage 3] Generating with constrained decoding (λ={lam}, beams={NUM_BEAMS}) …")
    else:
        print("[Stage 3] Generating with standard sampling …")

    explanation = generate_explanation(
        text=input_text,
        predicted_class=predicted_class,
        feature_data=feature_data,
        user_category=USER_CATEGORY,
        tokenizer=llm_tokenizer,
        model=llm_model,
        generator=generator,
    )
    print("\n── Generated explanation ─────────────────────────────────────")
    print(explanation)
    print("─────────────────────────────────────────────────────────────")

    # ── Stage 4 : Metrics ────────────────────────────────────────────────────────
    print("\n[Stage 4] Computing metrics …")
    read_metrics = readability_metrics(explanation)
    cov = lime_coverage(explanation, feature_data)
    hit = ontology_hit_rate(feature_data)

    result = {
        "input_index":           idx,
        "experiment_tag":        EXPERIMENT_TAG,
        "xai_method":            XAI_METHOD,
        "user_category":         USER_CATEGORY,
        "ablation_mode":         ABLATION_MODE,
        "constrained_decoding":  USE_CONSTRAINED_DECODING,
        "lambda":                LAMBDA_MAP.get(USER_CATEGORY, None),
        "ner_merging":           False,
        "predicted_class":       predicted_class,
        "confidence":            confidence,
        "text_snippet":          input_text[:120] + "…",
        "explanation":           explanation,
        "lime_coverage":         cov,
        "ontology_hit_rate":     hit,
        **read_metrics,
    }
    all_results.append(result)
    print(f"  ✅ Input {idx} done — FRE={read_metrics.get('flesch_reading_ease', 'N/A'):.2f}  "
          f"FKGL={read_metrics.get('flesch_kincaid_grade', 'N/A'):.2f}  "
          f"coverage={cov:.2f}  hit_rate={hit:.2f}")

print(f"\n✅ Batch complete — {len(all_results)} input(s) processed.")


  Processing input 1/6
  Preview: Endometriosis associated with massive ascites and absence of pelvic peritoneum. …

[Stage 1] Running IG feature attribution …
[Stage 1] Top 6 IG features:
  ascites                    score=+0.2454
  endometriosis              score=+0.2021
  peritoneum                 score=+0.1096
  massive                    score=+0.0978
  oophorectomy               score=+0.0946
  pelvic                     score=+0.0921

[Stage 2] Classifying text …
  Predicted class : Digestive system diseases
  Confidence      : 0.6624

[Stage 2] Mapping features to ontology ancestors …
  Ontology hits : ['ascites', 'endometriosis', 'peritoneum'] (3/6 features)
    ascites              → ['symptom', 'abdominal symptom', 'ascites']
    endometriosis        → ['disease of anatomical entity', 'reproductive system disease', 'female reproductive system disease', 'endometriosis']
    peritoneum           → ['multicellular anatomical structure', 'multi-tissue structure', 'serous memb

KeyboardInterrupt: 

## 5. Results summary

In [ ]:
# Summary table for all inputs
df = pd.DataFrame(all_results)

metric_cols = [
    "input_index", "xai_method", "user_category", "ablation_mode",
    "constrained_decoding", "ner_merging", "predicted_class", "confidence",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "lime_coverage", "ontology_hit_rate",
]
pd.set_option("display.max_colwidth", 40)
display(df[[c for c in metric_cols if c in df.columns]])

,input_index,xai_method,user_category,ablation_mode,constrained_decoding,ner_merging,predicted_class,confidence,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
0,1,IG,BEGINNER,normal,True,False,Digestive system diseases,0.6624,73.757281,6.118421,8.841846,0.6667,1.0
1,2,IG,BEGINNER,normal,True,False,Cardiovascular diseases,0.9097,54.553000,10.320723,12.340627,1.0000,1.0
2,3,IG,BEGINNER,normal,True,False,General pathological conditions,0.5573,-10.330000,18.675000,18.243606,0.0000,0.0
3,4,IG,BEGINNER,normal,True,False,Nervous system diseases,0.6343,58.867681,10.662938,11.208143,0.0000,1.0
4,5,IG,BEGINNER,normal,True,False,Neoplasms,0.8244,60.614550,10.932703,12.457976,1.0000,1.0
5,6,IG,BEGINNER,normal,True,False,Digestive system diseases,0.6946,42.419167,12.642500,14.554593,1.0000,1.0


In [ ]:
avg_fre = df["flesch_reading_ease"].mean()
avg_fkgl = df["flesch_kincaid_grade"].mean()

print(f"\nAverage Flesch Reading Ease: {avg_fre:.2f}")
print(f"Average Flesch-Kincaid Grade Level: {avg_fkgl:.2f}")


Average Flesch Reading Ease: 46.65
Average Flesch-Kincaid Grade Level: 11.56


## 6. Save final results

In [ ]:
EXPERIMENT_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(EXPERIMENT_RESULTS_PATH, index=False)
print(f"✅ Results saved → '{EXPERIMENT_RESULTS_PATH}' ({len(df)} row(s))")

# Print each explanation in full
for _, row in df.iterrows():
    print(f"\n── Input {int(row['input_index'])} ────────────────────────────────────────────────────")
    print(f"  Class      : {row['predicted_class']} (conf={row['confidence']:.4f})")
    print(f"  User       : {row['user_category']}")
    print(f"  Ablation   : {row['ablation_mode']}")
    print(f"  Tag        : {row['experiment_tag']}")
    print("─────────────────────────────────────────────────────────────")
    print(row['explanation'])

✅ Results saved → 'Results_w_CD\beginner_normal_ig.csv' (6 row(s))

── Input 1 ────────────────────────────────────────────────────
  Class      : Digestive system diseases (conf=0.6624)
  User       : BEGINNER
  Ablation   : normal
  Tag        : beginner_normal_ig
─────────────────────────────────────────────────────────────
In the abstract, the term "massive ascites" stands out as a key influencer. Ascites, a build-up of fluid in the腹腔, is a well-known symptom of certain diseases, such as cirrhosis. The phrase "absence of [某器官的] [某结构]" (即，"无[某器官]的[某结构]") 提供了额外的上下文，暗示了腹腔内结构的破坏或缺失，这可能是导致大量腹水的直接原因。此外，提及的"失败的医学抑制"和"腹腔切除术"进一步支持了对消化系统疾病的预测，因为这些手术通常与消化道疾病相关。因此，模型通过识别与消化系统疾病相关的特征（如腹水和腹腔结构破坏）来做出预测。

── Input 2 ────────────────────────────────────────────────────
  Class      : Cardiovascular diseases (conf=0.9097)
  User       : BEGINNER
  Ablation   : normal
  Tag        : beginner_normal_ig
─────────────────────────────────────────────────────────────
In the given abstract, the key words "